# Meeting 10, Act 2: An Image Is an Array

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 10.

A picture on a screen is also a block of numbers. In this act your group reads one picture as an array with two dimensions, then as an array with three, and computes new pictures from it.

## How this lab works

- **This is the notebook of Act 2 of meeting 10.** It starts from its own setup cell and uses nothing that was computed in the notebook of Act 1, except the number that your group drew there.
- **Work in the same group.** Each member opens this notebook and runs it. First open the **File** menu and choose **Save a copy in Drive**, so that your changes are kept.
- **Run the cells in order**, from the top. To run a cell, click it and press **Shift + Enter**.
- A paragraph that begins with **Your turn** asks you to do something before you go on.
- A code cell under the word **Check** tests your work and prints a message. If the message reports a problem, fix it before you go on.
- In Steps 13 and 15 you may ask an **AI assistant** to write code. Paste the prompt you used into the cell marked **Our prompt**, so that the notebook records how the code was made.
- **In Step 15** your group posts two pictures on the discussion board of this meeting. **At the end of the meeting** it posts once more, and each member uploads the notebooks to Canvas. The last section of this notebook says how.
- **After the meeting**, **Review Quiz 10** on Canvas asks about this lab. Where the notebook says that the quiz asks about a point, read that part again before you take the quiz.

**Your turn.** Double-click this cell and replace this line with the names of the members of your group.

In [ ]:
# Setup. Run this cell first.
import time
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# The address of the course's files for this unit, and of the folder that holds the images.
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
IMAGES = SITE + "images/"


def read_image(file):
    """Read an image file into a NumPy array with values from 0 to 1."""
    return np.array(Image.open(urlopen(IMAGES + file))) / 255.0


def show(pictures, titles):
    """Draw arrays as pictures: an array with two dimensions in grey, one with three in colour."""
    columns = 2 if len(pictures) == 4 else len(pictures)
    lines = len(pictures) // columns
    fig, axes = plt.subplots(lines, columns, figsize=(5 * columns, 4.2 * lines), squeeze=False)
    for ax, picture, title in zip(axes.flat, pictures, titles):
        ax.imshow(np.clip(picture, 0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


def average_time(function, data, repeats=5):
    """The average time of one call function(data), in seconds: the function of Step 9 in Act 1."""
    total = 0.0
    for _ in range(repeats):
        start = time.perf_counter()              # the clock before the call
        function(data)
        total += time.perf_counter() - start     # the time that this call took
    return total / repeats


print("Setup is done.")

## <span style="color:#1967d2">Step 11: A grey image is a matrix</span>

A digital image is a grid of small squares called **pixels**. In a **grey image** every pixel holds one number, its brightness: 0 is black, 1 is white, and the numbers between them are shades of grey. A grey image with H rows and W columns of pixels is therefore a **matrix** with H rows and W columns. In Act 1 a row of the matrix was one record; here a row is one horizontal line of pixels.

The course keeps thirty images, numbered like the datasets of Act 1. Your group's image has the number that your group drew in Act 1. If your group has no number, choose any number from 1 to 30.

**Your turn.** Every member types the group's number in the next cell, in place of the `0`, and runs the cell.

The cell reads the grey version of your image. The file stores each brightness as a whole number from 0 to 255; dividing by 255, which `read_image` does, brings it to the range from 0 to 1.

In [ ]:
IMAGE = 0        # replace 0 with your group's number, from 1 to 30

assert 1 <= IMAGE <= 30, "Type your group's number in place of the 0."

images = pd.read_csv(IMAGES + "images.csv", index_col="number")
picture_info = images.loc[IMAGE]
print(f"Image {IMAGE}: {picture_info['title']} ({picture_info['creator']})")
print(f"Source: {picture_info['source']}")

grey = read_image(picture_info["grey_file"])
H, W = grey.shape
print(f"shape of grey: {grey.shape}, which is H = {H} rows and W = {W} columns of pixels, {H * W:,} numbers")
print(f"darkest value {grey.min():.2f}, brightest value {grey.max():.2f}, mean {grey.mean():.2f}")

The next cell draws the array as a picture. Beside it, it draws a block of 12 by 12 pixels from the middle of the image, enlarged so that single pixels can be seen, and it prints the numbers of that block.

`grey[r:r + 12, c:c + 12]` selects a part of the matrix. `a:b` means the positions a, a + 1, and so on up to b − 1.

In [ ]:
r, c = H // 2, W // 2                    # the middle of the image
block = grey[r:r + 12, c:c + 12]         # 12 rows and 12 columns

show([grey, block], [f"grey: {H} by {W} pixels", f"12 by 12 pixels, from row {r} and column {c}"])
print("The numbers of the block, rounded to two decimals:")
print(np.round(block, 2))

**How to read the pictures.** The numbers along the edges are positions of rows and of columns. Row 0 is at the top, because a picture is drawn in the order in which the rows of the array are stored. In the enlarged block, each square is one pixel, and its shade is one number of the printed table: the larger the number, the lighter the square.

## <span style="color:#1967d2">Step 12: A colour image is an array with three dimensions</span>

In a colour image every pixel holds **three** numbers: how much red, how much green and how much blue light it gives, each from 0 to 1. The image is an array with three dimensions, of shape H by W by 3:

- `image[i, j]` holds the three numbers of the pixel in row i and column j;
- `image[:, :, 0]` is the **red channel**: a matrix with H rows and W columns that holds the red number of every pixel. A colon alone means all positions along that dimension;
- `image[:, :, 1]` is the green channel, and `image[:, :, 2]` is the blue channel.

An array with any number of dimensions is also called a **tensor**. A matrix is a tensor with two dimensions.

In [ ]:
image = read_image(picture_info["file"])
print(f"shape of image: {image.shape}, which is {image.size:,} numbers")
print("the pixel in row 0 and column 0:", np.round(image[0, 0], 2), "(red, green, blue)")

red, green, blue = image[:, :, 0], image[:, :, 1], image[:, :, 2]
print("shape of one channel:", red.shape)
print(f"mean of each channel: red {red.mean():.2f}, green {green.mean():.2f}, blue {blue.mean():.2f}")

show([image, red, green, blue], ["the colour image", "red channel", "green channel", "blue channel"])

Each channel is a matrix, so it is drawn in grey: a pixel is light in the picture of the red channel where the image holds much red.

**Your turn.** Find a region of your image that is light in one channel and dark in another. What colour does that region have in the colour image? Answer in the next cell, in one sentence.

**Our answer.** Double-click this cell and replace this line with your answer.

## <span style="color:#1967d2">Step 13: From colour to grey, in two ways</span>

A grey image can be computed from a colour image. A common rule gives every pixel the brightness

$$\text{grey} = 0.299 \cdot \text{red} + 0.587 \cdot \text{green} + 0.114 \cdot \text{blue}.$$

In words: 0.299 times the red number, plus 0.587 times the green number, plus 0.114 times the blue number. The three weights add to 1. Green has the largest weight because the eye is most sensitive to green light. The grey image of Step 11 was made from your colour image with this rule.

**Your turn.** Ask an AI assistant for **two** functions, in one prompt. Your prompt must state these requirements:

- `grey_loops(image)` uses two nested `for` loops, over the rows and over the columns. For each pixel it reads the three numbers `image[i, j, 0]`, `image[i, j, 1]` and `image[i, j, 2]`, and it stores the brightness in a new array created with `np.empty((H, W))`;
- `grey_vectorized(image)` computes the same result with no `for` loop, by arithmetic on whole channels;
- both take an array of shape (H, W, 3) and return an array of shape (H, W), with the three weights given above.

Paste your prompt into the next cell, and the two functions into the code cell after it.

**Our prompt.** Double-click this cell and replace this line with the prompt you gave to the AI assistant.

In [ ]:
# Your turn: paste the functions grey_loops and grey_vectorized here, then run this cell.

**Check.** The next cell compares both results with the grey image of Step 11. It then measures both functions with `average_time`, the function of Step 9 in Act 1, which the setup cell of this notebook defines again.

In [ ]:
assert "grey_loops" in globals() and "grey_vectorized" in globals(), (
    "Paste the functions grey_loops and grey_vectorized into the cell above and run that cell.")

for function in (grey_loops, grey_vectorized):
    result = np.asarray(function(image), dtype=float)
    assert result.shape == (H, W), f"{function.__name__} returned an array of shape {result.shape}; it must be {(H, W)}."
    worst = np.abs(result - grey).max()
    assert worst < 0.005, (
        f"{function.__name__} differs from the grey image of Step 11 by up to {worst:.3f}. "
        "Check the three weights and the order of the channels.")
print("Correct: both functions reproduce the grey image of Step 11.")

t_grey_loops = average_time(grey_loops, image, 3)
t_grey_vectorized = average_time(grey_vectorized, image, 20)
print()
print(f"Image {IMAGE}: {H} by {W} pixels")
print(f"{'version':14s}{'seconds per call':>18s}{'nanoseconds per pixel':>24s}")
print(f"{'loops':14s}{t_grey_loops:18.4f}{t_grey_loops / (H * W) * 1e9:24.1f}")
print(f"{'vectorized':14s}{t_grey_vectorized:18.4f}{t_grey_vectorized / (H * W) * 1e9:24.1f}")
print(f"The loops take {t_grey_loops / t_grey_vectorized:.0f} times as long as the vectorized version.")

## <span style="color:#1967d2">Step 14: One function applied to every entry</span>

In Act 1 the standard deviation needed the square of every distance from the mean, and the loop computed `d * d` for one entry at a time. On an array, `grey ** 2` squares **every** entry in one operation. An operation that applies the same function to each entry separately is called **pointwise**.

A pointwise operation on a grey image gives a new image of the same shape. Every brightness lies between 0 and 1, so:

- `grey ** 2` makes every value smaller, except 0 and 1, which stay as they are: the picture becomes darker;
- `np.sqrt(grey)` makes every value larger, except 0 and 1: the picture becomes lighter;
- `1 - grey` exchanges dark and light: the picture becomes its negative.

In [ ]:
darker = grey ** 2
lighter = np.sqrt(grey)
negative = 1 - grey

print(f"mean brightness: grey {grey.mean():.2f}, grey ** 2 {darker.mean():.2f}, "
      f"np.sqrt(grey) {lighter.mean():.2f}, 1 - grey {negative.mean():.2f}")
show([grey, darker, lighter, negative], ["grey", "grey ** 2", "np.sqrt(grey)", "1 - grey"])

## <span style="color:#1967d2">Step 15: Your own transformation</span>

**Your turn.** Make a new picture from your colour image by computing with its channels. Complete the function `our_transform` in the next code cell, yourselves or with an AI assistant. The function takes the array `image` and returns an array with the same height and width. Values below 0 or above 1 are drawn as 0 or 1.

Some starting points; you are not limited to them:

- exchange two channels, or set one channel to 0;
- apply a pointwise operation of Step 14 to one channel only;
- replace one channel by the grey image.

When the picture is what you want:

1. write in the cell marked **Our prompt** the prompt you used, or that you wrote the function yourselves;
2. **post the two pictures now** on the discussion board of this meeting, with one sentence that says what the transformation does;
3. keep the prompt and the function. **Review Quiz 10** asks for them, and asks you to explain what the function does to the array.

**Our prompt.** Double-click this cell and replace this line with your prompt, or with a note that you wrote the function yourselves.

In [ ]:
def our_transform(image):
    new = image.copy()               # a copy, so that image stays as it was
    # Your turn: change the array new here.
    return new


new_picture = our_transform(image)
assert new_picture.shape[:2] == (H, W), "The new picture must have the same height and width as the image."
show([image, new_picture], ["the image", "our transformation"])

## <span style="color:#1967d2">Step 16: Arrays with more dimensions</span>

The data of Acts 1 and 2 had two or three dimensions:

| Data | Number of dimensions | What the dimensions count |
|---|---|---|
| A table of Act 1 | 2 | records, features |
| A grey image | 2 | rows of pixels, columns of pixels |
| A colour image | 3 | rows of pixels, columns of pixels, channels |

**Your turn.** Name a kind of data that is naturally an array with **four** dimensions, and say what each of the four dimensions counts. Answer in the next cell. Review Quiz 10 returns to this question.

**Our answer.** Double-click this cell and replace this line with your answer.

## <span style="color:#1967d2">Step 17: When one operation is not enough</span>

Every new picture in this act came from arithmetic on whole arrays, and the loop over the pixels ran inside NumPy.

**Your turn.** Think of a transformation of an image that would be interesting to see and that you **cannot** write as a pointwise operation or as arithmetic on whole channels. For example, the new value of a pixel may depend on the pixels around it, or on values that were computed earlier for other pixels. Describe your transformation in the next cell, in one or two sentences.

Then discuss in your group: when a computation cannot be vectorized, is a slow Python loop the only choice? Review Quiz 10 asks about this.

**Our answer.** Double-click this cell and replace this line with the description of your transformation.

## At the end of the meeting: post and hand in

**The group posts** on the discussion board of this meeting. During Act 2 you posted the two pictures of Step 15. At the end of the meeting, add one more post, from the notebook of Act 1. It contains:

- the line that the cell under the timing chart of Step 9 printed;
- the timing chart of Step 9, and the two histograms of Step 8 for a column that your group finds interesting. To save a figure, right-click it and choose **Save image as**;
- one sentence on what surprised your group.

Under every image that you post, write **one sentence that says what the image shows**, so that a classmate who cannot see the image still gets the result.

Then read the posts of two other groups. Their datasets have other sizes: compare their nanoseconds per number with yours.

**Each member hands in both notebooks.** In the notebook of Act 1 and in this one, open the **File** menu, choose **Download**, then **Download .ipynb**. Upload the two files to this meeting's class-work assignment on Canvas.

**After the meeting**, take **Review Quiz 10** on Canvas. Some of its questions ask for an answer of a few sentences. Your notebooks, with your group's measurements, prompts and pictures in them, are the reference for them.